In [1]:
# ══════════════════════════════════════════════
# CELL 1 — Setup
# ══════════════════════════════════════════════
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import joblib
import numpy as np
from sklearn.metrics import (
    roc_auc_score, f1_score,
    precision_score, recall_score,
    accuracy_score
)
import warnings
warnings.filterwarnings('ignore')

# Saved models load karo
xgb_model = joblib.load('../model/best_model.pkl')
rf_model  = joblib.load('../model/rf_model.pkl')
lr_model  = joblib.load('../model/lr_model.pkl')

# Data load karo
X_train = np.load('../model/X_train_scaled.npy')
X_test  = np.load('../model/X_test_scaled.npy')
y_train = np.load('../model/y_train_smote.npy')
y_test  = np.load('../model/y_test.npy')

# MLflow configure karo
# Sab experiments ek SQLite DB mein save honge
mlflow.set_tracking_uri("sqlite:///./mlflow.db")
mlflow.set_experiment("Credit_Risk_Prediction")

print("✅ MLflow ready!")
print("✅ Sab models load ho gaye!")

C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/04/05 23:52:48 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/05 23:52:48 INFO mlflow.store.db.utils: Updating database tables
2026/04/05 23:52:52 INFO mlflow.tracking.fluent: Experiment with name 'Credit_Risk_Prediction' does not exist

✅ MLflow ready!
✅ Sab models load ho gaye!


In [2]:
# ══════════════════════════════════════════════
# CELL 2 — Teen Models Log Karo MLflow Mein
# ══════════════════════════════════════════════

# MLflow kya karta hai:
# - Har experiment run ka record rakhta hai
# - Parameters, metrics, aur models save karta hai
# - Different runs compare kar sakte ho UI mein

models_to_log = {
    "Logistic_Regression": lr_model,
    "Random_Forest":        rf_model,
    "XGBoost":              xgb_model,
}

for model_name, model in models_to_log.items():
    print(f"\nLogging {model_name}...")

    with mlflow.start_run(run_name=model_name):

        # Predictions
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        # Metrics calculate karo
        auc       = roc_auc_score(y_test, y_prob)
        f1        = f1_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall    = recall_score(y_test, y_pred)
        accuracy  = accuracy_score(y_test, y_pred)

        # MLflow mein log karo
        # mlflow.log_metric — numbers store karta hai
        mlflow.log_metric("auc_roc",   auc)
        mlflow.log_metric("f1_score",  f1)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall",    recall)
        mlflow.log_metric("accuracy",  accuracy)

        # mlflow.log_param — settings/config store karta hai
        mlflow.log_param("model_type",      model_name)
        mlflow.log_param("test_size",       "20%")
        mlflow.log_param("smote_applied",   True)
        mlflow.log_param("scaler",          "StandardScaler")
        mlflow.log_param("train_samples",   X_train.shape[0])

        # Model bhi save karo MLflow mein
        if model_name == "XGBoost":
            mlflow.xgboost.log_model(model, "xgboost_model")
        else:
            mlflow.sklearn.log_model(model, f"{model_name}_model")

        # Charts bhi log karo
        mlflow.log_artifact('../reports/roc_curve.png')
        mlflow.log_artifact('../reports/confusion_matrix.png')

        print(f"  ✅ {model_name}: AUC={auc:.4f}, F1={f1:.4f}, Precision={precision:.4f}, Recall={recall:.4f}")

print("\n✅ Sab models MLflow mein log ho gaye!")


Logging Logistic_Regression...


2026/04/05 23:52:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/05 23:52:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  ✅ Logistic_Regression: AUC=0.8035, F1=0.2758, Precision=0.1705, Recall=0.7222

Logging Random_Forest...


2026/04/05 23:53:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/05 23:53:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  ✅ Random_Forest: AUC=0.8396, F1=0.3325, Precision=0.2201, Recall=0.6798

Logging XGBoost...


2026/04/05 23:53:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  ✅ XGBoost: AUC=0.8107, F1=0.2091, Precision=0.1186, Recall=0.8818

✅ Sab models MLflow mein log ho gaye!


In [3]:
# ══════════════════════════════════════════════
# CELL 3 — Best Run Dhundo
# ══════════════════════════════════════════════

# Programmatically best model find karo
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name("Credit_Risk_Prediction")
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.auc_roc DESC"]  # AUC ke hisaab se sort karo
)

print("Top 3 Runs (AUC-ROC se sort kiya):")
print(f"\n{'Run Name':<25} {'AUC-ROC':<10} {'F1':<10} {'Precision':<12} {'Recall':<10}")
print("-" * 67)
for run in runs[:3]:
    name      = run.data.tags.get('mlflow.runName', 'Unknown')
    auc       = run.data.metrics.get('auc_roc', 0)
    f1        = run.data.metrics.get('f1_score', 0)
    precision = run.data.metrics.get('precision', 0)
    recall    = run.data.metrics.get('recall', 0)
    print(f"{name:<25} {auc:<10.4f} {f1:<10.4f} {precision:<12.4f} {recall:<10.4f}")

best_run = runs[0]
print(f"\n🏆 Best Model: {best_run.data.tags.get('mlflow.runName')}")
print(f"   AUC-ROC: {best_run.data.metrics['auc_roc']:.4f}")

Top 3 Runs (AUC-ROC se sort kiya):

Run Name                  AUC-ROC    F1         Precision    Recall    
-------------------------------------------------------------------
Random_Forest             0.8396     0.3325     0.2201       0.6798    
XGBoost                   0.8107     0.2091     0.1186       0.8818    
Logistic_Regression       0.8035     0.2758     0.1705       0.7222    

🏆 Best Model: Random_Forest
   AUC-ROC: 0.8396


In [4]:
# ══════════════════════════════════════════════
# CELL 4 — MLflow UI Launch karo
# ══════════════════════════════════════════════

print("MLflow UI start karne ke liye terminal mein yeh run karo:")
print("\n  mlflow ui --backend-store-uri sqlite:///./mlflow.db")
print("\nPhir browser mein jaao: http://localhost:5000")
print("\nWahan tum:")
print("  - Sare runs compare kar sakte ho")
print("  - Charts dekh sakte ho")
print("  - Models download kar sakte ho")
print("  - Parameters filter kar sakte ho")

MLflow UI start karne ke liye terminal mein yeh run karo:

  mlflow ui --backend-store-uri sqlite:///./mlflow.db

Phir browser mein jaao: http://localhost:5000

Wahan tum:
  - Sare runs compare kar sakte ho
  - Charts dekh sakte ho
  - Models download kar sakte ho
  - Parameters filter kar sakte ho


In [5]:
# ══════════════════════════════════════════════
# CELL 5 — Autolog (Automatic Tracking)
# ══════════════════════════════════════════════

# Agar manually log karna nahi chahte, autolog use karo
# Yeh automatically sab parameters aur metrics log kar leta hai

print("XGBoost Autolog Example:")

mlflow.xgboost.autolog()  # Sab automatically log hoga!

with mlflow.start_run(run_name="XGBoost_AutoLog"):
    # Re-train karo — autolog sab pakad lega
    from xgboost import XGBClassifier
    auto_model = XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=42
    )
    auto_model.fit(X_train, y_train)
    # Manually sirf AUC log karo
    y_prob_auto = auto_model.predict_proba(X_test)[:, 1]
    mlflow.log_metric("auc_roc", roc_auc_score(y_test, y_prob_auto))

print("✅ AutoLog run complete! MLflow UI mein dekho.")

XGBoost Autolog Example:


2026/04/05 23:53:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


✅ AutoLog run complete! MLflow UI mein dekho.
